In [ ]:
%load_ext autoreload
%autoreload 2

import hydra

from genpp import BASE_DIR
from genpp.configs import register_resolvers

try:
    register_resolvers()
except ValueError:
    pass

# Test if Model works in general

In [ ]:
with hydra.initialize_config_dir(version_base=None, config_dir=str(BASE_DIR / "configs")):
    cfg = hydra.compose(config_name="base_chen_spatial")

In [ ]:
datamodule = hydra.utils.instantiate(cfg.data.module)
datamodule.prepare_data()
datamodule.setup("fit")
dataloader = datamodule.train_dataloader()

In [ ]:
model = hydra.utils.instantiate(
    cfg.model,
    rescaler=datamodule.y_reverseModules if cfg.model.use_rescaler else None,
    internal_td_scaling="learned",
)

In [ ]:
trainer = hydra.utils.instantiate(
    cfg.trainer,
    logger=None,
    detect_anomaly=False,
    accelerator="gpu",
    max_epochs=10,
    overfit_batches=10,
    check_val_every_n_epoch=1,
)
trainer.fit(model, datamodule=datamodule)

## Try to load the model from checkpoint

In [ ]:
import os
from pathlib import Path

from genpp.models.cgm.chen import CNNChenModel

cwd = Path(os.getcwd())
checkpoint_path = list((cwd / "lightning_logs" / "version_1" / "checkpoints").glob("*.ckpt"))[0]
checkpoint_path

In [ ]:
for batch in dataloader:
    break

In [ ]:
mod_new = CNNChenModel.load_from_checkpoint(checkpoint_path)

In [ ]:
def move_to_device(batch, device):
    """
    Recursively moves all tensors inside a nested dict to the given device.
    """
    if isinstance(batch, dict):
        return {k: move_to_device(v, device) for k, v in batch.items()}

    # If it has .to(), treat it as a tensor-like object
    if hasattr(batch, "to"):
        return batch.to(device)

    return batch  # leave other objects as-is


batch = move_to_device(batch, mod_new.device)

In [ ]:
res = mod_new.predict_step(batch)

## Some random short tests

In [ ]:
import torch
from einops import rearrange

from genpp.models.loss import PatchwiseEnergyScore

In [ ]:
size = 30
channels = 1
beta = 1.0
t_c = torch.randint(0, 10, (64, 10, channels, size, size)) * 1.0
t_res = torch.randint(0, 10, (64, channels, size, size)) * 10.0

for p in range(3, 22, 2):
    pes = PatchwiseEnergyScore(height=size, width=size, patch_size=p, beta=beta)

    t_c_flat = rearrange(t_c, "b n c h w -> b n (c h w)")
    t_res_flat = rearrange(t_res, "b c h w -> b (c h w)")
    res = pes(t_c_flat, t_res_flat, mode="complete")
    res /= p**beta

    print(res.mean())